# Results for model: mistralai/mistral-large-3-675b-instruct-2512

In [ ]:
import polars as pl
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

data_path = './jane_street_data/train.parquet'

df = pl.read_parquet(data_path)

top_quantile_threshold = df.select(
    pl.col('feature_00').quantile(0.85).over('date_id').alias('top_quantile')
).unique()

df = df.join(top_quantile_threshold, on='date_id')

df = df.with_columns(
    pl.when(pl.col('feature_00') >= pl.col('top_quantile'))
    .then(1)
    .otherwise(0)
    .alias('feature_00_top_quantile')
)

features = [col for col in df.columns if col.startswith('feature_') or col == 'feature_00_top_quantile']
target = 'responder_6'

X = df.select(features).to_numpy()
y = df.select(target).to_numpy().ravel()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=50
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

val_preds = model.predict(X_val)
mse = mean_squared_error(y_val, val_preds)
print(f'Validation MSE: {mse}')